## Backstage.io

Zuerst hollen wir uns die Default Konfiguration von backstage, dazu erstellen wir eine Dummy Installation

In [ ]:
%%bash
cat <<EOF >values.yaml

postgresql:
  enabled: true

ingress:
  enabled: true
EOF

helm repo add backstage https://backstage.github.io/charts
helm repo update
helm upgrade -i dummy-backstage backstage/backstage \
  --namespace backstage-system \
  --create-namespace \
  -f values.yaml


Und kopieren das `/app` Verzeichnis auf die VM

In [ ]:
%%bash
rm -rf app
mkdir app
kubectl exec -n backstage-system deployment/dummy-backstage -- tar cf - . | tar xf - -C ./app

Dann deinstallieren wie die Dummy Installation

In [ ]:
%%bash
helm uninstall dummy-backstage  --namespace backstage-system

### Dev Installation - Variante a)

Jetzt haben wir Konfigurationsdateien für Backstage

In [ ]:
%%bash
ls -lh app
cat app/app-config.production.yaml

Daraus erstellen wir eine ConfigMap und übergeben die via `values.yaml`

In [ ]:
%%bash
cat <<EOF >app/app-config-extra.yaml
catalog:
  # Overrides the default list locations from app-config.yaml as these contain example data.
  # See https://backstage.io/docs/features/software-catalog/#adding-components-to-the-catalog for more details
  # on how to get entities into the catalog.
  locations:
    # Local example data, replace this with your production config, these are intended for demo use only.
    # File locations are relative to the backend process, typically in a deployed context, such as in a Docker container, this will be the root
    - type: file
      target: /app/examples/entities.yaml

    # Local example template
    - type: file
      target: /app/examples/template/template.yaml
      rules:
        - allow: [Template]
EOF


In [ ]:
%%bash
kubectl create configmap dev-app-config --from-file=app/app-config-extra.yaml --namespace backstage-system 

In [ ]:
%%bash
cat <<EOF >dev-values.yaml
backstage:
  extraAppConfig:
    - configMapRef: dev-app-config
      filename: app-config-extra.yaml
      
postgresql:
  enabled: true

ingress:
  enabled: true
EOF

helm repo add backstage https://backstage.github.io/charts
helm repo update
helm upgrade -i dev-backstage backstage/backstage \
  --namespace backstage-system \
  --create-namespace \
  -f dev-values.yaml
  

### Dev Installation - Variante b)

In [ ]:
%%bash
cat <<EOF >app/app-config.yaml
app:
  title: Mein Backstage
  baseUrl: http://localhost:7007

backend:
  baseUrl: http://localhost:7007
  listen: 0.0.0.0:7007

catalog:
  locations:
    - type: file
      target: /app/examples/entities.yaml
EOF

In [ ]:
%%bash
kubectl create configmap app-config --from-file=app/app-config.yaml --namespace backstage-system 

In [ ]:
%%bash
cat <<EOF >dev-values.yaml
backstage:
  extraAppConfig:
    - configMapRef: app-config
      filename: app-config.yaml
      
postgresql:
  enabled: true

ingress:
  enabled: true
EOF

helm repo add backstage https://backstage.github.io/charts
helm repo update
helm upgrade -i dev-backstage backstage/backstage \
  --namespace backstage-system \
  --create-namespace \
  -f dev-values.yaml

Der URL ist

In [ ]:
%%bash
echo "http://$(cat ~/work/server-ip)"

- - - 

## Aufträge

### Links

* [Backstage Helm Chart](https://github.com/backstage/charts)
* [Adding components to the catalog](https://backstage.io/docs/features/software-catalog/#adding-components-to-the-catalog)

---
### Aufräumen

In [ ]:
%%bash
helm uninstall dev-backstage  --namespace backstage-system
kubectl delete configmap dev-app-config --namespace backstage-system 